# Causal Inference Crash Course:
## Regression Discontinuity Design
Julian Hsu

## Overview
* This continues the causal inference series with **regression discontinuity design (RDD)** — a design-based approach that finds a *local natural experiment* already sitting in your data.
* Unlike the cross-sectional models (OLS, propensity, DML) that lean on **unconfoundedness**, RDD leans on a **continuity** assumption at a cutoff.
* This presentation will cover:
    * When you can use RDD, and the big idea
    * The identifying assumption and estimand (a *local* treatment effect)
    * Sharp vs. fuzzy designs
    * Modern estimation: local polynomials, bandwidth, and robust inference
    * Validating the design (manipulation, covariate, and placebo tests)
    * A worked example with simulated data
* Intended audience: a scientist comfortable with the earlier decks (potential outcomes, OLS, inference).

## A familiar setup: a threshold-based policy
* Suppose customers receive a **promotion** ($W_i$) once an **eligibility score** ($X_i$) crosses a cutoff $c$.
    * e.g., a spend threshold that unlocks free shipping, a loyalty-points tier, a credit-score cutoff for an offer.
* We want the effect of the promotion on **next-quarter purchases** ($Y_i$).
* Customers *just below* and *just above* the cutoff are nearly identical — except one group got the promotion.
* That threshold is the natural experiment we will exploit.

## The Big Idea
![Image](Figures/RD_Figure_0.png)
* Treatment is assigned by a rule: $W_i = 1\{X_i \ge c\}$. The **jump** in $Y$ at the cutoff is the treatment effect.

## Framing with potential outcomes
* Recall from the Foundations deck $Y_i = Y_i(0) + \tau_i\, W_i$, now with $W_i = 1\{X_i \ge c\}$.
* We *cannot* use unconfoundedness here: $X_i$ **perfectly determines** $W_i$, so there is no overlap — no treated and control units share the same $X_i$.
* Instead, RDD compares the two sides *in the limit* at the cutoff:
$$ \tau_{RD} = \lim_{x \downarrow c} E[Y_i \mid X_i = x] \;-\; \lim_{x \uparrow c} E[Y_i \mid X_i = x] $$
* This is the average treatment effect **for units at the cutoff**.

## The identifying assumption: continuity
* **Continuity**: $E[Y_i(0)\mid X_i=x]$ and $E[Y_i(1)\mid X_i=x]$ are *continuous* in $x$ at $c$.
    * In words: *absent the treatment*, nothing else jumps at the cutoff. The only reason $Y$ can jump is the treatment itself.
* Contrast with unconfoundedness: we do **not** need to observe and control every confounder — we only need confounders to vary *smoothly* through the cutoff.
* The threat is **manipulation / sorting**: if units precisely control $X_i$ to land just above $c$, the two groups differ and continuity breaks.

## What RDD estimates: a *local* effect
* $\tau_{RD}$ is the effect **at the cutoff**, for the subpopulation with $X_i \approx c$.
* Strong internal validity (credible, transparent), but limited external validity:
    * It need not equal the effect for units far from $c$.
    * With heterogeneous effects (see the HTE deck), extrapolate away from $c$ only with a good argument.
* Practical implication: you need enough data *near* the cutoff, and you are answering a question *about* the cutoff.

# Sharp vs. Fuzzy
Does crossing the cutoff *guarantee* treatment, or just make it more likely?

## Sharp vs. fuzzy designs
* **Sharp RD**: crossing the cutoff *deterministically* assigns treatment, $W_i = 1\{X_i \ge c\}$.
* **Fuzzy RD**: crossing the cutoff *changes the probability* of treatment — imperfect compliance.
![Image](Figures/RD_Figure_5.png)

## Fuzzy RD is an IV / Wald ratio
* Scale the jump in the outcome by the jump in the treatment probability:
$$ \tau_{FRD} = \frac{\lim_{x\downarrow c} E[Y|X=x] - \lim_{x\uparrow c} E[Y|X=x]}{\lim_{x\downarrow c} E[W|X=x] - \lim_{x\uparrow c} E[W|X=x]} $$
* This is exactly the **instrumental-variables (Wald)** estimator, using $1\{X_i \ge c\}$ as an instrument for $W_i$.
* It identifies a LATE for **compliers** at the cutoff — units treated *because* they crossed.

# Estimation
The whole game is a careful local regression near the cutoff.

## Don't fit a global high-order polynomial
![Image](Figures/RD_Figure_1.png)
* A global polynomial uses data *far* from the cutoff to fit *at* the cutoff, and the estimate **swings with the order** (Gelman & Imbens 2019). Prefer a local fit.

## Estimate locally, with a kernel
* Keep only observations within a **bandwidth** $h$ of the cutoff, and fit a **local linear** (or low-order local polynomial) regression on each side, weighting by a kernel $K(\cdot)$:
$$ \hat\tau = \hat\alpha_r - \hat\alpha_l $$
where $\hat\alpha_r, \hat\alpha_l$ are the fitted intercepts *at* $c$ from the right- and left-side local regressions.
* Local **linear** is the standard default; higher orders can lower bias but add variance.
* A **triangular** kernel (more weight near $c$) is MSE-optimal at the boundary.

## Bandwidth is the key tuning choice
![Image](Figures/RD_Figure_2.png)
* Small $h$ &rarr; less bias, more variance. Large $h$ &rarr; more bias, less variance.
* Use a data-driven **MSE-optimal** bandwidth (Calonico, Cattaneo, Titiunik 2014), not a hand-picked one.

## Robust bias-corrected inference
* The MSE-optimal bandwidth is *deliberately* large enough that some bias remains — so the conventional confidence interval is **mis-centered** and under-covers.
* The modern fix (CCT 2014) is **robust bias-corrected (RBC)** inference:
    1. estimate and subtract the leading bias, and
    2. widen the standard error to account for having estimated that bias.
* This is what `rdrobust` reports by default &mdash; use the **Robust** interval, not the conventional one.

## Always show the RD plot
![Image](Figures/RD_Figure_6.png)
* Bin the running variable, plot the mean of $Y$ in each bin, and overlay the local fits.
* The plot makes the jump &mdash; and any funny business &mdash; visible *before* any formal test.

# Validating the design
The estimate is only as good as the continuity assumption. So test it.

## The design is testable &mdash; unusually so
* Continuity is an assumption, but it has **observable implications** we can check:
    1. The **density** of $X_i$ should not jump at $c$ (no manipulation).
    2. **Predetermined covariates** should not jump at $c$ (local balance).
    3. The effect should vanish at **placebo cutoffs**.
    4. Results should be stable to the **bandwidth** and to a **donut** around $c$.
* This is the RDD analog of the toolkit in the *Arguable Validation* deck.

## Test 1: manipulation (density) test
![Image](Figures/RD_Figure_3.png)
* If units sort across the cutoff, the density of $X_i$ **jumps** at $c$.
* Formal test: McCrary (2008); the modern local-polynomial version is in `rddensity` (Cattaneo, Jansson, Ma 2020). Null: the density is continuous.

## Test 2: covariate continuity (placebo outcomes)
![Image](Figures/RD_Figure_4.png)
* A **predetermined** covariate is a placebo outcome: its "effect" at the cutoff should be zero.
* Re-run the RD with each covariate as the outcome; a jump signals a broken design. (Just like the placebo tests in the *Arguable Validation* deck.)

## Tests 3&ndash;5: placebo cutoffs, donut, sensitivity
* **Placebo cutoffs**: estimate the "effect" at fake cutoffs where no policy changes. You should find nothing.
* **Donut hole**: drop observations *right at* the cutoff and re-estimate. If the estimate moves a lot, the units most likely to have sorted are driving it.
* **Bandwidth sensitivity**: report the estimate across a range of $h$ (as in the bandwidth figure). A credible result is not knife-edge.

# Worked example
Simulated data, so we know the right answer ($\tau = 5$).

## Simulation setup
* Companion notebook [here](https://github.com/shoepaladin/causalinference_crashcourse/blob/main/Notebooks/7%20Regression%20Discontinuity.ipynb).
* Retail DGP: eligibility score $X_i \in [0,100]$, cutoff $c=50$, promotion $W_i=1\{X_i\ge c\}$, outcome $Y_i$ = next-quarter purchases, with a true effect $\tau=5$ (same convention as Foundations).
* Three scenarios:
    1. **Clean** &mdash; continuity holds.
    2. **Nonlinear** &mdash; the baseline $E[Y(0)\mid X]$ is curved.
    3. **Manipulation** &mdash; high-value customers sort just above the cutoff.
* Four estimators: naive mean difference, global polynomial (order 4), a hand-coded local linear, and `rdrobust` (RBC).

## Bias across estimators and scenarios
* Mean bias $=(\hat\tau - \tau)$ over 500 simulated datasets; $\tau=5$.

| Estimator | Clean | Nonlinear | Manipulation |
|:---|:---:|:---:|:---:|
| Naive mean difference | +2.28 | +5.33 | +2.60 |
| Global polynomial (order 4) | +0.01 | -0.04 | +1.99 |
| Local linear (hand-coded) | +0.02 | +0.05 | +1.73 |
| `rdrobust` (robust bias-corrected) | +0.00 | -0.06 | +1.89 |

* Local methods recover $\tau$ even under curvature; the naive difference does not.

## Coverage &mdash; and why validation comes first
* 95% confidence-interval coverage over the same simulations:

| Estimator | Clean | Nonlinear | Manipulation |
|:---|:---:|:---:|:---:|
| Local linear (naive SE) | 0.95 | 0.94 | 0.00 |
| `rdrobust` (RBC) | 0.94 | 0.95 | 0.00 |

* Under **manipulation**, *every* estimator is biased and coverage collapses to ~0% &mdash; no amount of clever estimation rescues a broken design. That is why the validation tests come first.

## Conclusion
* RDD turns a threshold rule into a credible **local natural experiment**.
* Strengths: transparent identification, a *testable* assumption, and compelling visuals.
* Costs: a **local** effect at the cutoff (external validity), a need for data near $c$, and it fails if the running variable is manipulated.
* Reach for RDD when a consequential decision hinges on a cutoff &mdash; and always run the validation battery.

# Appendix Slides

## Appendix: the kink design (RKD)
* Sometimes the *level* of treatment is continuous at $c$, but its **slope** changes &mdash; e.g., a benefit formula whose marginal rate changes at a threshold.
* The **regression kink design** identifies the effect from the change in the *slope* of $Y$ in $X$ at $c$, rather than a jump in level.
* Same continuity logic, same `rdrobust` machinery (with `deriv=1`).

## Appendix: other extensions
* **Multiple cutoffs / multi-score RD**: units face different thresholds, or a boundary in two dimensions (e.g., geographic RD).
* **Discrete running variables**: few mass points near $c$ require care &mdash; the local randomization framework (Cattaneo, Idrobo, Titiunik 2024) is designed for this.
* **Local randomization**: treat a small window around $c$ as a randomized experiment and use randomization inference (`rdlocrand`).

## Appendix: the local-linear estimator
* On each side of the cutoff, solve the weighted least-squares problem:
$$ \min_{\alpha,\beta}\; \sum_i K\!\left(\frac{X_i-c}{h}\right)\left(Y_i - \alpha - \beta(X_i - c)\right)^2 $$
* $\hat\alpha$ is the fitted level *at* the cutoff; $\hat\tau = \hat\alpha_r - \hat\alpha_l$.
* RBC replaces the point estimate and variance with bias-corrected versions computed with a second (pilot) bandwidth $b \ge h$.

## Literature Review of Related Papers
* **Foundational**
    * Thistlethwaite & Campbell (1960). "Regression-Discontinuity Analysis." *J. Educational Psychology.*
    * Hahn, Todd, Van der Klaauw (2001). "Identification and Estimation of Treatment Effects with a RD Design." *Econometrica.* [link](https://www.jstor.org/stable/2692190)
    * Lee (2008). "Randomized experiments from non-random selection in U.S. House elections." [link](https://www.sciencedirect.com/science/article/abs/pii/S0304407607001121)
    * Imbens & Lemieux (2008). "Regression discontinuity designs: A guide to practice." [link](https://www.nber.org/papers/w13039)
* **Estimation & inference**
    * Calonico, Cattaneo, Titiunik (2014). "Robust Nonparametric Confidence Intervals for RD Designs." *Econometrica.* [link](https://onlinelibrary.wiley.com/doi/10.3982/ECTA11757)
    * Calonico, Cattaneo, Farrell (2020). "Optimal bandwidth choice for robust bias-corrected inference in RD designs." [link](https://academic.oup.com/ectj/article/23/2/192/5698educations)
    * Gelman & Imbens (2019). "Why High-Order Polynomials Should Not Be Used in RD Designs." [link](https://www.tandfonline.com/doi/full/10.1080/07350015.2017.1366909)
* **Practical guides**
    * Cattaneo, Idrobo, Titiunik (2020, 2024). "A Practical Introduction to RD Designs: Foundations / Extensions." Cambridge. [link](https://rdpackages.github.io/references/Cattaneo-Idrobo-Titiunik_2020_CUP.pdf)
* **Validation & software**
    * McCrary (2008). "Manipulation of the running variable in the RD design: A density test." [link](https://www.sciencedirect.com/science/article/abs/pii/S0304407607001133)
    * Cattaneo, Jansson, Ma (2020). "Simple Local Polynomial Density Estimators." (`rddensity`) [link](https://nppackages.github.io/references/Cattaneo-Jansson-Ma_2020_JASA.pdf)
    * Software: `rdrobust`, `rddensity`, `rdlocrand` &mdash; [rdpackages.github.io](https://rdpackages.github.io)

In [ ]:
# --- Figure generation (run this notebook to regenerate Figures/RD_Figure_*.png) ---
# All figures below are produced from a simulated retail RD design.
import os
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(20260702)
CUT, TAU = 50.0, 5.0
ACCENT, ACCENT2, CTRLC = "#0e7490", "#0f766e", "#94a3b8"
FIGDIR = os.path.join(os.getcwd(), "Figures")
os.makedirs(FIGDIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 11, "axes.grid": True, "grid.alpha": .25})


def baseline(x, shape="linear"):
    xc = (x - CUT) / 10.0
    if shape == "linear":
        return 10.0 + 0.8 * xc
    return 10.0 + 0.8 * xc + 2.6 * np.sin(0.9 * xc)  # "nonlinear": high curvature


def gen(n=2000, shape="linear", manipulate=False, fuzzy=False, seed=None):
    rng = np.random.default_rng(seed) if seed is not None else RNG
    x = np.clip(rng.normal(CUT, 18, n), 0, 100)
    ability = rng.normal(0, 1, n)                      # unobserved outcome driver
    if manipulate:                                     # high-ability units sort above c
        below = (x > CUT - 6) & (x < CUT)
        move = below & (ability > 0.2) & (rng.random(n) < 0.75)
        x[move] = CUT + rng.uniform(0, 4, move.sum())
    if fuzzy:
        w = (rng.random(n) < np.where(x >= CUT, 0.75, 0.20)).astype(float)
    else:
        w = (x >= CUT).astype(float)
    y = baseline(x, shape) + TAU * w + 2.0 * ability + rng.normal(0, 1.5, n)
    z = 0.5 + 0.03 * (x - CUT) + 0.8 * ability + rng.normal(0, 0.6, n)  # predetermined covariate
    return x, w, y, z


def naive(x, w, y):
    return y[w == 1].mean() - y[w == 0].mean()


def global_poly(x, w, y, order=4):
    xc = x - CUT
    feats = [np.ones_like(xc), w]
    for k in range(1, order + 1):
        feats += [xc ** k, w * xc ** k]
    beta, *_ = np.linalg.lstsq(np.column_stack(feats), y, rcond=None)
    return beta[1]


def local_linear(x, w, y, h=8.0, return_se=False):
    xc = x - CUT
    ints, ses = {}, {}
    for side, mask in (("r", w == 1), ("l", w == 0)):
        xs, ys = xc[mask], y[mask]
        sel = np.abs(xs) <= h
        xs, ys = xs[sel], ys[sel]
        wgt = np.maximum(1 - np.abs(xs / h), 0)                     # triangular kernel
        X = np.column_stack([np.ones_like(xs), xs]); W = np.diag(wgt)
        XtW = X.T @ W
        beta = np.linalg.solve(XtW @ X, XtW @ ys)
        ints[side] = beta[0]
        resid = ys - X @ beta
        s2 = (wgt * resid ** 2).sum() / max(wgt.sum() - 2, 1)
        cov = np.linalg.inv(XtW @ X) @ (XtW @ W @ X) @ np.linalg.inv(XtW @ X) * s2
        ses[side] = np.sqrt(max(cov[0, 0], 0))
    tau = ints["r"] - ints["l"]
    return (tau, np.sqrt(ses["r"] ** 2 + ses["l"] ** 2)) if return_se else tau


def _save(fig, name):
    fig.tight_layout(); fig.savefig(os.path.join(FIGDIR, name), bbox_inches="tight"); plt.close(fig)


def _band(ax):
    ax.axvline(CUT, color="#334155", ls="--", lw=1.2)

In [ ]:
# Figure 0 — the sharp discontinuity and the true conditional expectation
x, w, y, z = gen(1400, "linear", seed=7)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.scatter(x[w == 0], y[w == 0], s=8, color=CTRLC, alpha=.5, label="Not eligible (W=0)")
ax.scatter(x[w == 1], y[w == 1], s=8, color=ACCENT, alpha=.5, label="Eligible (W=1)")
gx0 = np.linspace(0, CUT, 100); gx1 = np.linspace(CUT, 100, 100)
ax.plot(gx0, baseline(gx0), color="#0f172a", lw=2)
ax.plot(gx1, baseline(gx1) + TAU, color="#0f172a", lw=2)
ax.plot(gx1, baseline(gx1), color="#0f172a", lw=1, ls=":")
ax.annotate(f"jump = $\\tau$ = {TAU:.0f}", xy=(CUT, baseline(CUT) + TAU / 2),
            xytext=(CUT + 12, baseline(CUT) - 4), color=ACCENT2, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=ACCENT2))
_band(ax)
ax.set_xlabel("Eligibility score  X   (cutoff c = 50)"); ax.set_ylabel("Next-quarter purchases  Y ($)")
ax.legend(loc="upper left", fontsize=8, framealpha=.9)
_save(fig, "RD_Figure_0.png")

In [ ]:
# Figure 1 — global polynomials are sensitive to order (Gelman & Imbens 2019)
x, w, y, z = gen(2500, "nonlinear", seed=11)
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.4))
ax = axes[0]; ax.scatter(x, y, s=6, color="#cbd5e1", alpha=.5); _band(ax)
gx0 = np.linspace(0, CUT, 200); gx1 = np.linspace(CUT, 100, 200)
for gx, mask in ((gx0, x < CUT), (gx1, x >= CUT)):
    ax.plot(gx, np.polyval(np.polyfit(x[mask], y[mask], 1), gx), color="#f59e0b", lw=2, ls="--")
    ax.plot(gx, np.polyval(np.polyfit(x[mask], y[mask], 6), gx), color="#b91c1c", lw=2)
for gx, mask, col in ((gx0, x < CUT, "#0f172a"), (gx1, x >= CUT, ACCENT2)):
    m = mask & (np.abs(x - CUT) <= 8); gg = gx[np.abs(gx - CUT) <= 8]
    ax.plot(gg, np.polyval(np.polyfit(x[m], y[m], 1), gg), color=col, lw=3)
ax.plot([], [], color="#f59e0b", ls="--", label="global order 1")
ax.plot([], [], color="#b91c1c", label="global order 6")
ax.plot([], [], color=ACCENT2, lw=3, label="local linear")
ax.legend(fontsize=7.5, loc="upper left"); ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title("One dataset, three fits", fontsize=10)
orders = range(1, 8); mean, sd = [], []
for o in orders:
    b = [global_poly(*gen(3000, "nonlinear", seed=300 + r)[:3], o) for r in range(120)]
    mean.append(np.mean(b)); sd.append(np.std(b))
ll = [local_linear(*gen(3000, "nonlinear", seed=300 + r)[:3], 8.0) for r in range(120)]
ax = axes[1]
ax.errorbar(list(orders), mean, yerr=sd, color="#b91c1c", marker="o", ms=4, capsize=3, label="global polynomial")
ax.axhline(np.mean(ll), color=ACCENT2, lw=2, label="local linear")
ax.axhline(TAU, color="#0f172a", ls="--", lw=1.2, label=f"true $\\tau$={TAU:.0f}")
ax.set_xlabel("global polynomial order"); ax.set_ylabel("estimated  $\\hat\\tau$")
ax.set_title("Estimate swings with the order", fontsize=10); ax.legend(fontsize=7.5)
_save(fig, "RD_Figure_1.png")

In [ ]:
# Figure 2 — bandwidth / bias-variance tradeoff
x, w, y, z = gen(4000, "nonlinear", seed=5)
hs = np.arange(3, 31, 1.0); est, lo, hi = [], [], []
for h in hs:
    t, se = local_linear(x, w, y, h, return_se=True)
    est.append(t); lo.append(t - 1.96 * se); hi.append(t + 1.96 * se)
fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.fill_between(hs, lo, hi, color=ACCENT, alpha=.15)
ax.plot(hs, est, color=ACCENT, lw=2, marker="o", ms=3, label="Local-linear $\\hat\\tau$")
ax.axhline(TAU, color="#0f172a", ls="--", lw=1.2, label=f"true $\\tau$={TAU:.0f}")
try:                                   # MSE-optimal bandwidth from rdrobust, if installed
    from rdrobust import rdrobust
    hopt = float(rdrobust(y=y, x=x, c=CUT).bws.loc["h", "left"])
    ax.axvline(hopt, color=ACCENT2, ls=":", lw=1.5, label=f"MSE-opt h≈{hopt:.1f}")
except Exception:
    pass
ax.set_xlabel("bandwidth  h"); ax.set_ylabel("estimated  $\\hat\\tau$"); ax.legend(fontsize=8)
_save(fig, "RD_Figure_2.png")

In [ ]:
# Figure 3 — running-variable density: clean vs. manipulated
xc, *_ = gen(6000, "linear", seed=2)
xm, *_ = gen(6000, "linear", manipulate=True, seed=2)
fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.2), sharey=True)
for ax, data, title in zip(axes, [xc, xm], ["No manipulation", "Sorting at the cutoff"]):
    ax.hist(data[data < CUT], bins=np.linspace(0, CUT, 26), color=CTRLC, alpha=.85)
    ax.hist(data[data >= CUT], bins=np.linspace(CUT, 100, 26), color=ACCENT, alpha=.85)
    _band(ax); ax.set_title(title, fontsize=10); ax.set_xlabel("X")
axes[0].set_ylabel("count")
_save(fig, "RD_Figure_3.png")

In [ ]:
# Figure 4 — covariate continuity placebo (a predetermined covariate should not jump)
x, w, y, z = gen(3000, "linear", seed=9)
fig, ax = plt.subplots(figsize=(6.2, 3.4))
bins = np.linspace(0, 100, 26); idx = np.digitize(x, bins)
for side, col in ((0, CTRLC), (1, ACCENT)):
    m = w == side
    bx = [x[m & (idx == b)].mean() for b in range(1, len(bins)) if (m & (idx == b)).any()]
    bz = [z[m & (idx == b)].mean() for b in range(1, len(bins)) if (m & (idx == b)).any()]
    ax.scatter(bx, bz, s=26, color=col, edgecolor="white", zorder=3)
for mask, col in ((x < CUT, "#0f172a"), (x >= CUT, ACCENT2)):
    xs = np.sort(x[mask]); ax.plot(xs, np.polyval(np.polyfit(x[mask], z[mask], 1), xs), color=col, lw=2)
_band(ax)
ax.set_xlabel("Eligibility score  X"); ax.set_ylabel("Predetermined covariate  Z")
ax.set_title("Placebo: a pre-treatment covariate should NOT jump", fontsize=9.5)
_save(fig, "RD_Figure_4.png")

In [ ]:
# Figure 5 — fuzzy RD: treatment probability jumps, but not from 0 to 1
x, w, y, z = gen(4000, "linear", fuzzy=True, seed=4)
fig, ax = plt.subplots(figsize=(6.2, 3.4))
bins = np.linspace(0, 100, 26); idx = np.digitize(x, bins)
bx = [x[idx == b].mean() for b in range(1, len(bins)) if (idx == b).any()]
bw = [w[idx == b].mean() for b in range(1, len(bins)) if (idx == b).any()]
ax.scatter(bx, bw, s=26, color=ACCENT, edgecolor="white", zorder=3); _band(ax)
ax.set_ylim(-.05, 1.05); ax.set_xlabel("Eligibility score  X"); ax.set_ylabel("P(received promotion)")
ax.annotate("jump < 1\n(imperfect compliance)", xy=(CUT, .5), xytext=(CUT + 8, .32),
            fontsize=8, color=ACCENT2, arrowprops=dict(arrowstyle="->", color=ACCENT2))
ax.set_title("Fuzzy RD: treatment probability jumps, but not from 0 to 1", fontsize=9.5)
_save(fig, "RD_Figure_5.png")

In [ ]:
# Figure 6 — the RD plot: binned means + local linear fits
x, w, y, z = gen(3000, "linear", seed=3)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
bins = np.linspace(0, 100, 26); idx = np.digitize(x, bins)
for side, col in ((0, CTRLC), (1, ACCENT)):
    m = w == side
    bx = [x[m & (idx == b)].mean() for b in range(1, len(bins)) if (m & (idx == b)).any()]
    by = [y[m & (idx == b)].mean() for b in range(1, len(bins)) if (m & (idx == b)).any()]
    ax.scatter(bx, by, s=28, color=col, edgecolor="white", zorder=3)
for mask, col in ((x < CUT, "#0f172a"), (x >= CUT, ACCENT2)):
    xs = np.sort(x[mask]); ax.plot(xs, np.polyval(np.polyfit(x[mask], y[mask], 1), xs), color=col, lw=2)
_band(ax)
ax.set_xlabel("Eligibility score  X"); ax.set_ylabel("Binned mean of  Y")
ax.set_title("RD plot: binned means + local linear fits", fontsize=10)
_save(fig, "RD_Figure_6.png")